In [1]:
# Measure the time it takes every cell to run
%pip install ipython-autotime
%load_ext autotime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.0 MB/s eta 0:00:00
time: 257 µs (started: 2026-03-15 19:34:16 +00:00)


In [2]:
# Set up idc-index, used to make clinical table parameters human-readable
%pip install idc-index
from idc_index import IDCClient
c = IDCClient()
c.fetch_index('clinical_index')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 65.1 MB/s eta 0:00:00
  Attempting uninstall: duckdb
    Found existing installation: duckdb 1.3.2
    Uninstalling duckdb-1.3.2:
      Successfully uninstalled duckdb-1.3.2
time: 25.8 s (started: 2026-03-15 19:34:22 +00:00)


In [3]:
# Set up BigQuery, used to construct patient cohort of interest
import os
my_ProjectID = "nlst-radiomics"
os.environ["GCP_PROJECT_ID"] = my_ProjectID

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
bq_client = bigquery.Client(my_ProjectID)

time: 14.7 s (started: 2026-03-15 19:34:53 +00:00)


In [30]:
# First, what are the possible segmentation types?
# Note: I checked that 'AIMI lung and nodule AI segmentation' contains 'Lung' and 'Nodule' segment labels

seg_query = """
SELECT SeriesDescription
FROM `bigquery-public-data.idc_v23.dicom_all`
WHERE
  collection_id = 'nlst'
  AND Modality = 'SEG'
"""

seg_result = bq_client.query(seg_query)
seg_metadata_df = seg_result.result().to_dataframe()

print(f"\n--- Unique Series Descriptions ---")
display(seg_metadata_df['SeriesDescription'].value_counts().head(20))

print(f"\n--- Unique Series Descriptions Containing 'lung and nodule' ---")
lung_nodule_segmentation_types = seg_metadata_df[seg_metadata_df['SeriesDescription'].str.contains('lung and nodule', case=False)]['SeriesDescription'].unique().tolist()
display(lung_nodule_segmentation_types)


--- Unique Series Descriptions ---


,count
SeriesDescription,
TotalSegmentator(v1.5.6) Segmentation of Series 2,47987
TotalSegmentator(v1.5.6) Segmentation of Series 3,38198
TotalSegmentator(v1.5.6) Segmentation of Series 4,13362
TotalSegmentator(v1.5.6) Segmentation of Series 5,6180
TotalSegmentator(v1.5.6) Segmentation of Series 6,5377
TotalSegmentator(v1.5.6) Segmentation of Series 1,4449
TotalSegmentator(v1.5.6) Segmentation of Series 0,1132
AIMI lung and nodule AI segmentation,1042
3d_fullres-tta_nnU-Net_Segmentation,1039



--- Unique Series Descriptions Containing 'lung and nodule' ---


['AIMI lung and nodule radiologist 4 corrected segmentation',
 'AIMI lung and nodule radiologist 5 corrected segmentation',
 'AIMI lung and nodule radiologist 8 corrected segmentation',
 'AIMI lung and nodule AI segmentation']

time: 2.07 s (started: 2026-03-15 21:01:52 +00:00)


In [55]:
# How many CT/SEG pairs with lung and nodule segmentations are there?
# Note: If seg.SeriesDescription is not radiologist corrected, there are 1042 SEG files with lung and nodule segmentations

ct_seg_join_query = f"""
SELECT
  DISTINCT seg.SeriesInstanceUID AS SEG_SeriesInstanceUID,
  ct.PatientID,
  ct.SeriesInstanceUID AS CT_SeriesInstanceUID,
  seg.SeriesDescription AS SEG_SeriesDescription,
  seg.StudyDate,
  CASE
    WHEN FORMAT_DATE('%Y', seg.StudyDate) LIKE '1999%' THEN 'T0'
    WHEN FORMAT_DATE('%Y', seg.StudyDate) LIKE '2000%' THEN 'T1'
    WHEN FORMAT_DATE('%Y', seg.StudyDate) LIKE '2001%' THEN 'T2'
    ELSE 'Other'
  END AS StudyYear
FROM
  `bigquery-public-data.idc_v23.dicom_all` AS ct
JOIN
  `bigquery-public-data.idc_v23.dicom_all` AS seg
ON
  ct.collection_id = seg.collection_id
  AND ct.PatientID = seg.PatientID
WHERE
  ct.collection_id = 'nlst'
  AND ct.Modality = 'CT'
  AND seg.Modality = 'SEG'
  AND seg.SeriesDescription IN ({', '.join([f"'{s}'" for s in lung_nodule_segmentation_types])})
  AND EXISTS (
    SELECT 1
    FROM UNNEST(seg.ReferencedSeriesSequence) AS ref_seq
    WHERE ct.SeriesInstanceUID = ref_seq.SeriesInstanceUID
  )
"""

ct_seg_join_result = bq_client.query(ct_seg_join_query)
ct_seg_join_df = ct_seg_join_result.result().to_dataframe()

print(f"Number of distinct patients with a CT/SEG pair: {len(ct_seg_join_df['PatientID'].unique())}")
print(f"Number of unique CT series: {len(ct_seg_join_df['CT_SeriesInstanceUID'].unique())}")
print(f"Number of CT/SEG pairs: {ct_seg_join_df.shape[0]}")
print(f"Number of distinct study dates: {len(ct_seg_join_df['StudyDate'].unique())}")

Number of distinct patients with a CT/SEG pair: 572
Number of unique CT series: 1042
Number of CT/SEG pairs: 1144
Number of distinct study dates: 3
time: 3.82 s (started: 2026-03-15 21:53:50 +00:00)


In [56]:
# Two SEG series can correspond to one CT series because of radiologist corrected segmentations. "Keep" the CT/SEG pair with the radiologist corrected segmentation and throw out the other.

priority_segmentation_type = 'AIMI lung and nodule AI segmentation'

# Create a priority column: 0 for non-AI, 1 for AI
ct_seg_join_df['priority'] = ct_seg_join_df['SEG_SeriesDescription'].apply(lambda x: 1 if x == priority_segmentation_type else 0)

# Sort by CT_SeriesInstanceUID and then by priority (0 comes before 1, so non-AI is prioritized)
ct_seg_join_df_filtered = ct_seg_join_df.sort_values(by=['CT_SeriesInstanceUID', 'priority'])

# Drop duplicates, keeping the first (which will be the non-AI one if present, otherwise the first AI one)
ct_seg_join_df_one_to_one = ct_seg_join_df_filtered.drop_duplicates(subset='CT_SeriesInstanceUID', keep='first')

print(f"\nNumber of CT/SEG pairs after filtering: {ct_seg_join_df_one_to_one.shape[0]}")
display(ct_seg_join_df_one_to_one.head())


Number of CT/SEG pairs after filtering: 1042


,SEG_SeriesInstanceUID,PatientID,CT_SeriesInstanceUID,SEG_SeriesDescription,StudyDate,StudyYear,priority
839,1.2.276.0.7230010.3.1.3.17436516.3112497.17206...,107002,1.2.840.113654.2.55.10057224705584082952969157...,AIMI lung and nodule AI segmentation,1999-01-02,T0,1
256,1.2.276.0.7230010.3.1.3.17436516.3112767.17206...,107030,1.2.840.113654.2.55.10087518978221069034420730...,AIMI lung and nodule AI segmentation,1999-01-02,T0,1
769,1.2.276.0.7230010.3.1.3.17436516.3182127.17206...,126955,1.2.840.113654.2.55.10212803674068251180698456...,AIMI lung and nodule AI segmentation,2000-01-02,T1,1
479,1.2.276.0.7230010.3.1.3.17436516.3103212.17206...,104999,1.2.840.113654.2.55.10222078521568314836955069...,AIMI lung and nodule AI segmentation,1999-01-02,T0,1
173,1.2.276.0.7230010.3.1.3.17436516.3180509.17206...,126446,1.2.840.113654.2.55.10223655654547815709262155...,AIMI lung and nodule radiologist 5 corrected s...,1999-01-02,T0,0


time: 31.6 ms (started: 2026-03-15 21:53:58 +00:00)


In [62]:
# download CT/SEG pairs for 5 distinct patients using idc_index

five_malignant_patientIDs = ct_seg_join_df_one_to_one['PatientID'].unique().tolist()[:5]
print(f"Five malignant patient IDs: {five_malignant_patientIDs}")

patientID_to_check = five_malignant_patientIDs[0] # Using the first patient ID from the previously selected list

patient_pairs = []
for index, row in ct_seg_join_df_one_to_one[ct_seg_join_df_one_to_one['PatientID'] == patientID_to_check].iterrows():
    patient_pairs.append([row['CT_SeriesInstanceUID'], row['SEG_SeriesInstanceUID']])

print(f"CT/SEG pairs for PatientID {patientID_to_check}:")
display(patient_pairs)

c.download_from_selection(seriesInstanceUID= patient_pairs[0], downloadDir= f"/malignant_trial")


#for patientID in five_malignant_patientIDs:
  #ct_seg_patientID_df = ct_seg_join_df_one_to_one['PatientID'].equals(patientID)
  #download_directory = f"/malignant_trial/{patientID}"
  #c.download_from_selection(seriesInstanceUID=CT_SeriesInstanceUID, downloadDir=download_directory)


#c.download_from_selection(seriesInstanceUID= CT_SeriesInstanceUID, downloadDir= download_directory)

Five malignant patient IDs: ['107002', '107030', '126955', '104999', '126446']
CT/SEG pairs for PatientID 107002:


[['1.2.840.113654.2.55.10057224705584082952969157580585052644',
  '1.2.276.0.7230010.3.1.3.17436516.3112497.1720666652.959236'],
 ['1.2.840.113654.2.55.46585009632538178829747577506479558322',
  '1.2.276.0.7230010.3.1.3.17436516.3112628.1720666654.215702']]

time: 1.76 s (started: 2026-03-15 22:06:03 +00:00)


In [43]:
patientID_to_check = five_malignant_patientIDs[0] # Using the first patient ID from the previously selected list

patient_pairs = []
for index, row in ct_seg_join_df_one_to_one[ct_seg_join_df_one_to_one['PatientID'] == patientID_to_check].iterrows():
    patient_pairs.append((row['CT_SeriesInstanceUID'], row['SEG_SeriesInstanceUID']))

print(f"CT/SEG pairs for PatientID {patientID_to_check}:")
display(patient_pairs)

CT/SEG pairs for PatientID 107002:


[('1.2.840.113654.2.55.10057224705584082952969157580585052644',
  '1.2.276.0.7230010.3.1.3.17436516.3112497.1720666652.959236'),
 ('1.2.840.113654.2.55.46585009632538178829747577506479558322',
  '1.2.276.0.7230010.3.1.3.17436516.3112628.1720666654.215702')]

time: 7.85 ms (started: 2026-03-15 21:31:21 +00:00)
